[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/t04_integrals.ipynb)

# Which integration method? Five questions, and one habit

Integrate x eˣ. Most people write (x²/2)·eˣ + C. Differentiate that back and an extra piece appears, so it is wrong.
The right answer is x eˣ − eˣ + C, and the way to be sure of any answer is the same: **differentiate it back**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top. When a slider or a menu appears, move it and watch. The last
cell checks any answer of your own: type the integral and your answer into its two boxes, and press **Check**. It
works in your browser (no install) and in Google Colab. Nothing can break: if something looks odd, run everything
again from the top.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the tools. Run this box first.
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, FloatSlider, Dropdown, Text, Button, Output, VBox, HBox
from IPython.display import display
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas and integrals

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})
plt.close(plt.figure())                     # warms up the plotting: the first figure is slow in the browser

x = sp.symbols("x")
NAMES = {"x": x, "e": sp.E, "ln": sp.log, "log": sp.log, "pi": sp.pi, "sqrt": sp.sqrt,
         "asin": sp.asin, "atan": sp.atan, "arcsin": sp.asin, "arctan": sp.atan}
RULES = standard_transformations + (implicit_multiplication_application, convert_xor)

def read(s):
    """Turn what you type into maths: x^2 or x**2, e^x or exp(x), ln(x), sin(x), atan(x), sqrt(x)."""
    return parse_expr(s.replace("−", "-").replace("×", "*"), local_dict=NAMES, transformations=RULES)

def say(e):
    """Show an expression the way you'd write it: x^2 for x squared, e^x for e to the x."""
    return str(e).replace("**", "^").replace("exp(x)", "e^x").replace("exp(", "e^(").replace("log(", "ln(").replace("*", "·")

def slope_check(f, F, xs=None):
    """Differentiate F back and compare it with f. Returns (matches, extra piece)."""
    extra = sp.simplify(sp.diff(F, x) - f)
    return extra == 0, extra

def plot_check(f, F, title, lo=0.2, hi=3.0):
    """The integrand (thick) and the slope of the answer (dashed). If the answer is right, they lie on top of each other."""
    fx = sp.lambdify(x, f, "numpy"); dF = sp.lambdify(x, sp.diff(F, x), "numpy")
    xs = np.linspace(lo, hi, 300)
    fig, ax = plt.subplots(figsize=(8.5, 3.8))
    ax.plot(xs, fx(xs) * np.ones_like(xs), color=TEAL, lw=7, alpha=0.55, label="the integrand")
    ax.plot(xs, dF(xs) * np.ones_like(xs), color=CORAL, lw=2, ls="--", label="the slope of the answer")
    ax.set_xlabel("x"); ax.set_title(title, fontsize=13); ax.legend(loc="upper left")
    plt.show()

def check(integral, answer):
    """Differentiate the answer back and compare it with the integral."""
    f, F = read(integral), read(answer)
    ok, extra = slope_check(f, F)
    print(f"∫ {say(f)} dx  =  {say(F)} + C ?")
    print(f"  differentiate your answer back:  {say(sp.expand(sp.diff(F, x)))}")
    print("  ✓ matches" if ok else f"  ✗ does not match: an extra piece {say(extra)}")
    plot_check(f, F, "matches" if ok else "does not match")

def checker(integral=""):
    """The two boxes (the integral, your answer) and a Check button."""
    box_f = Text(value=integral, placeholder="x*e^x", description="integral")
    box_F = Text(value="", placeholder="your answer", description="answer")
    button = Button(description="Check")
    out = Output()
    def on_check(_):
        out.clear_output()
        with out:
            try: check(box_f.value, box_F.value)
            except Exception as err: print("I couldn't read that. Try e.g. x^2/2*e^x  (", err, ")")
    button.on_click(on_check)
    display(VBox([HBox([box_f, box_F, button]), out]))

## 1 · Differentiate it back

An integral asks: **which function has this slope?** So any answer can be tested by taking its slope. Pick an answer
for the integral of x eˣ. The thick line is x eˣ. The dashed line is the slope of the answer you picked. If they
don't sit on top of each other, the answer is wrong.

In [ ]:
f0 = x * sp.exp(x)
CANDIDATES = {"(x²/2)·eˣ   (the tempting one)": x**2 / 2 * sp.exp(x),
              "x eˣ − eˣ   (by parts)": x * sp.exp(x) - sp.exp(x)}

def try_answer(answer="(x²/2)·eˣ   (the tempting one)"):
    F = CANDIDATES[answer]
    ok, extra = slope_check(f0, F)
    plot_check(f0, F, "matches" if ok else f"does not match: extra piece {say(extra)}")

interact(try_answer, answer=Dropdown(options=list(CANDIDATES), description="answer"));

**Notice.** The tempting answer's slope runs above x eˣ, by exactly x²eˣ/2: the product rule's second piece. The
by-parts answer lands exactly on it.

## 2 · Why "+ C"

Shift an answer up or down by any constant C. Its slope doesn't change at all, so every shifted copy is also an
answer. That's why every indefinite integral ends with + C: the answer is a whole family of curves.

In [ ]:
def family(C=0.0):
    xs = np.linspace(-2, 2, 300)
    fig, ax = plt.subplots(figsize=(8.5, 4))
    for k in np.arange(-3, 3.5, 1.0):
        ax.plot(xs, xs * np.exp(xs) - np.exp(xs) + k, color=DIM, lw=1, alpha=0.6)
    ax.plot(xs, xs * np.exp(xs) - np.exp(xs) + C, color=LILAC, lw=3, label=f"x eˣ − eˣ + {C:.1f}")
    x1 = 1.0
    y1 = x1 * np.exp(x1) - np.exp(x1) + C
    s = x1 * np.exp(x1)                      # the slope there: x eˣ, whatever C is
    ax.plot([x1 - 0.5, x1 + 0.5], [y1 - 0.5 * s, y1 + 0.5 * s], color=SAGE, lw=3, label=f"slope at x = 1: {s:.3f}")
    ax.plot([x1], [y1], "o", color=CORAL, ms=9)
    ax.set_ylim(-5, 6); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title("move C: the curve shifts, its slope stays")
    plt.show()

interact(family, C=FloatSlider(value=0, min=-3, max=3, step=0.5, description="C"));

**Notice.** The coral point moves up and down, but the green slope line never tilts. Every C gives the same slope.

## 3 · The five questions, on the video's integrals

Ask them in order: **Do I already know it?** (a standard form) · **Is there an inside, with its slope outside?**
(substitution) · **Is it a product of two kinds?** (by parts: choose u by ILATE) · **Is it a fraction I can split?**
(partial fractions) · **Is there a quadratic that won't factor?** (complete the square). Then differentiate back.

Pick one of the video's integrals. You'll see which question it falls to, the answer, and the check.

In [ ]:
GUIDE = {
    "1/(x² + 9)":                    ("a standard form",       1 / (x**2 + 9),               sp.atan(x / 3) / 3),
    "cos x · e^(sin x)":             ("substitution",          sp.cos(x) * sp.exp(sp.sin(x)), sp.exp(sp.sin(x))),
    "x (x² + 1)⁵":                   ("substitution",          x * (x**2 + 1)**5,            (x**2 + 1)**6 / 12),
    "x eˣ":                          ("by parts, u = x",       x * sp.exp(x),                x * sp.exp(x) - sp.exp(x)),
    "x ln x":                        ("by parts, u = ln x",    x * sp.log(x),                x**2 / 2 * sp.log(x) - x**2 / 4),
    "eˣ sin x":                      ("by parts, twice",       sp.exp(x) * sp.sin(x),        sp.exp(x) * (sp.sin(x) - sp.cos(x)) / 2),
    "(x + 7)/((x − 1)(x + 3))":      ("partial fractions",     (x + 7) / ((x - 1) * (x + 3)), 2 * sp.log(x - 1) - sp.log(x + 3)),
    "1/(x²(x + 1))":                 ("partial fractions",     1 / (x**2 * (x + 1)),         -sp.log(x) - 1 / x + sp.log(x + 1)),
    "1/(x² + 4x + 13)":              ("complete the square",   1 / (x**2 + 4 * x + 13),      sp.atan((x + 2) / 3) / 3),
    "1/√(8 + 2x − x²)":              ("complete the square",   1 / sp.sqrt(8 + 2 * x - x**2), sp.asin((x - 1) / 3)),
}

def guide(integral="x eˣ"):
    how, f, F = GUIDE[integral]
    ok, _ = slope_check(f, F)
    print(f"∫ {integral} dx")
    print(f"  falls to:  {how}")
    print(f"  answer:    {say(F)} + C")
    print(f"  check:     differentiate back → {'matches' if ok else 'does NOT match'}")
    lo, hi = (1.2, 2.8) if "x − 1" in integral or "x²(x" in integral else (0.2, 2.8)
    plot_check(f, F, f"∫ {integral} dx: {how}", lo, hi)

interact(guide, integral=Dropdown(options=list(GUIDE), value="x eˣ", description="integral"));

**Notice.** Every answer passes the same test. When the logs need bars (ln|x − 1|), the check is drawn where the
insides are positive.

## 4 · Partial fractions, by computer

sympy's `apart` splits a fraction into the pieces you would write by hand. Compare them with the video's A, B and D.

In [ ]:
for top_bottom in [(x + 7) / ((x - 1) * (x + 3)), 1 / (x**2 * (x + 1)), 2 / ((x + 1) * (x**2 + 1))]:
    print(f"{say(top_bottom):>28}   =   {say(sp.apart(top_bottom, x))}")

**Notice.** Over x² + 1, which won't factor, the top is a line (−x + 1), one degree less than its bottom.

**The working that loses a mark.** ∫ x cos x dx = x sin x − cos x + C? Run the next cell. Differentiating back leaves
an extra 2 sin x: the slip was a sign. The right answer is x sin x + cos x + C.

In [ ]:
for answer in ["x*sin(x) - cos(x)", "x*sin(x) + cos(x)"]:
    ok, extra = slope_check(x * sp.cos(x), read(answer))
    print(f"{answer:>20}:  " + ("matches" if ok else f"does not match: extra piece {say(extra)}"))

## 5 · Check your answer

Type the integral (what comes before dx) in the first box and your answer in the second. Leave out the + C. Write
x squared as `x^2`, e to the x as `e^x`, the natural log as `ln(x)`, and use `*` or a space to multiply. Then press
**Check**. It differentiates your answer back and tells you whether it matches.

In [ ]:
# Two boxes and a button: type the integral and your answer, then press Check.
checker()

## The challenge
∫ ln x dx. There's no product. Or is there? Hint: ln x · 1. Try your answer in the check above.

## Sources
CBSE Mathematics 041 syllabus 2026–27 (Integrals). H. E. Kasube, "A Technique for Integration by Parts", *The
American Mathematical Monthly* 90 (1983), pp. 210–211 (LIATE). All problems are our own. Retrieved 2026-10-24.